# 語言基礎


## 1.**控制流程**

- 語法：
    - while 迴圈：只要條件為 True，就會一直重複執行裡面的代碼。
    - break 關鍵字：強行跳出（結束）當前整個迴圈。


In [ ]:
# 計數器到 3 就停
count = 1
while count <= 5:
    print(f"目前計數: {count}")
    if count == 3:
        print("遇到 break，立刻離開迴圈！")
        break  # 強行中斷
    count += 1

- 實作：
    - 在機台自動化中，我們不可能預知測試要跑多久，所以會用 while True 讓程式不停檢查機台狀態（稱為輪詢 Polling），直到收到「測試完成」或「機器當機」的信號，再用 break 結束等待。
    - 結合 while 輪詢（Polling）與 if/else 條件判斷，模擬機台狀態監控。
    - 腳本：
        - elif 若改用 if 會導致，當執行上面 if 時又同時檢查下面 if，效率較差。


In [ ]:
import time
import random


def get_tester_status():
    # 模擬產線測試機狀態：90% 處於 BUSY/READY，10% 觸發 DOWN 機
    return random.choice(["READY", "BUSY", "BUSY", "DOWN"])


# 實戰場景：自動化 Pipeline 必須輪詢等待機台釋放，若 DOWN 機則觸發警報
timeout_seconds = 10
start_time = time.time()

print("開始監控測試機狀態...")
while True:
    status = get_tester_status()
    print(f"當前機台狀態: {status}")

    if status == "READY":
        print("[進入通道] 機台已空閒，準備下發測試任務。")
        break
    elif status == "DOWN":
        print("[嚴重警報] 機台發生硬體故障 (DOWN)！中斷 Pipeline。")
        break

    # 逾時保護機制
    if time.time() - start_time > timeout_seconds:
        print("[逾時中斷] 等待機台釋放超時，請檢查排程系統。")
        break

    time.sleep(1)  # 每秒輪詢一次

## 2. try/except 例外處理（專注於設備連線與讀檔失敗）

- 語法：
    - 為了防止程式因為不可預期的錯誤（如除以零、檔案不存在）而直接崩潰。
    - try：把「可能會出錯」的程式碼放在這裡。
    - except：如果出錯了，要執行什麼補救措施（化解錯誤）。
    - else：如果 try 裡面「完全沒出錯」，才會執行的安全通道。
    - finally：不論最後是成功還是失敗，都一定會執行的清理收尾區塊。


In [ ]:
# 處理除以零的錯誤
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:  # 精準捕捉並處理特定的錯誤
        print("錯誤：分母不能為 0！")
        result = None  # 賦予空值
    else:
        print("計算成功！")
    finally:
        print("收尾：除法運算結束。")
    return result


print(safe_divide(10, 2))  # 成功通道
print(safe_divide(10, 0))  # 觸發攔截


- except 處理


In [ ]:
# 寫法一：精準捕捉不同的錯誤
try:
    # 可能出錯的程式碼
    result = 10 / int(user_input)
except ZeroDivisionError:
    print("錯誤：分母不能為零！")
except ValueError:
    print("錯誤：請輸入有效的整數數字！")

# 寫法二：用元組（Tuple）包在一起（統一處理多種錯誤）
try:
    # 執行某些檔案讀取或運算
    pass
except (ZeroDivisionError, TypeError, ValueError):
    print("發生了數值或型態相關的錯誤，請檢查輸入資料！")


- 實作：
    - 線上測試機（Tester）或晶圓探針台（Probe Card）因為網路波動突然連線中斷（Timeout），或者 MES 產出的 Log 檔被其他程序佔用導致讀取失敗。若不寫例外處理，自動化 Pipeline 就會崩潰中斷。
    - 腳本：
        - raise：拋出。
        - except Exception：程式停止。


In [ ]:
# 模擬元件：定義半導體測試設備常見的自訂例外 (Custom Exceptions)
class DeviceConnectionError(Exception):
    """設備網路中斷、握手(Handshake)失敗時拋出"""

    pass


class LogFileNotFoundError(Exception):
    """機台未正常生成當班日誌，或檔案路徑毀損時拋出"""

    pass


import random


def connect_to_probe_card():
    """模擬連線至探針卡控制器的函式"""
    if random.choice([True, False]):
        raise DeviceConnectionError(
            "ERROR: ProbeCard(IP: 192.168.1.50) connection timeout."
        )
    print("[SUCCESS] 順利連線至 Probe Card 控制器。")


# 實作例外處理
try:
    print("開始執行機台初始化程序...")
    connect_to_probe_card()

except DeviceConnectionError as e:
    # 針對設備連線失敗的被害者處置：記錄警告、通知警報系統、執行安全關閉
    print(f"[連線攔截] 設備層級錯誤: {e}")
    print("[安全對策] 啟動離線快取緩衝區，準備進行備用路由連線...")

except LogFileNotFoundError as e:
    # 針對日誌讀取失敗的處置
    print(f"[檔案攔截] 日誌層級錯誤: {e}")

except Exception as e:
    # 兜底捕獲其餘未知錯誤（如記憶體溢出、型別非預期錯誤）
    # 程式停止運行，並將錯誤訊息回報給開發團隊
    print(f"[未知崩潰] 系統發生非預期突發錯誤: {e}")

else:
    # 當 try 區塊「完全沒有發生例外」時才會執行的黃金通道
    print("[完美通關] 設備所有參數配置並初始化成功，準備下發測試 Recipe。")

finally:
    # 不管成功或失敗「都一定會執行」的清理現場
    print("[系統清理] 關閉暫存通訊埠(Socket Port)，釋放硬體控制權。")


## 3. 檔案與目錄自動化操作 (os, pathlib)

- 語法：
    - mkdir(parents=True, exist_ok=True)：建立資料夾。exist_ok=True 代表如果資料夾已經存在，不要報錯。
    - rglob("*.擴充檔名")：遞迴（包含子資料夾）搜尋所有符合標準的檔案。

In [ ]:
from pathlib import Path

# 建立資料夾並尋找檔案
my_folder = Path("./demo_folder")
my_folder.mkdir(exist_ok=True)  # 安全建立資料夾

# 假設裡面有檔案，用 rglob 把它們全部找出來
for file in my_folder.rglob("*.txt"):
    print(f"找到文字檔: {file.name}")


- 實作：
    - 產線每天會自動產出數千個資料夾與測試報告（如不同 Lot、Wafer 的 CSV、TXT 檔）。工程師需要用程式自動掃描特定路徑、建立備份資料夾、或依日期分類檔案。
    - 實作自動掃描產線日誌、自動建立備份資料夾並分類。
    - 腳本：
        - pathlib.Path.glob() 或 rglob()：遞迴搜尋資料夾內所有的特定格式檔案。
        - Path.mkdir(parents=True, exist_ok=True)：自動建立缺失的輸出目錄。
        - os.path.join() 與路徑跨平台相容處理。


In [ ]:
from pathlib import Path
import os
import shutil  # 處理高階的檔案與資料夾操作

# 1. 定義模擬產線原始資料夾與備份路徑
source_dir = Path("./mes_raw_logs")
backup_dir = Path("./archive_logs/2026-08-05")

# 建立模擬測試環境與檔案
# .touch() 建立一個全新的空白檔案
source_dir.mkdir(parents=True, exist_ok=True)
(source_dir / "LOT_A01_Wafer01.csv").touch()
(source_dir / "LOT_A01_Wafer02.txt").touch()
(source_dir / "summary.json").touch()

# 2. 自動化核心腳本
# 自動建立缺失的輸出備份目錄（parents=True 確保父目錄存在，exist_ok=True 防止重複建立報錯）
backup_dir.mkdir(parents=True, exist_ok=True)

print("開始遞迴掃描並備份 CSV 與 TXT 測試報告...")
# 使用 rglob 進行遞迴搜尋特定格式檔案
for file_path in source_dir.rglob("*"):
    if file_path.suffix in [".csv", ".txt"]:
        # 跨平台路徑安全拼接與複製
        dest_path = backup_dir / file_path.name
        shutil.copy(file_path, dest_path)
        print(f"成功備份: {file_path.name} -> {dest_path}")

# 清理模擬目錄
shutil.rmtree(source_dir)

## 4. 生成器/迭代器（處理大量日誌記憶體最佳化）

- 語法：
    - 一般函式用 return 回傳結果，回傳後函式就結束了。
    - 生成器則是使用 yield 關鍵字。它每次只回傳一個值，然後「暫停」在那裡，等下次被呼叫時再繼續往下跑。
    - 好處：不需要一次把幾百萬個資料放進記憶體，要一個才生一個（Lazy Evaluation 惰性求值）。


In [ ]:
# 建立一個簡單的數字產生器
def simple_counter():
    print("開始產生第一個數")
    yield 1
    print("開始產生第二個數")
    yield 2


# 使用 for 迴圈來跟生成器「要資料」
for number in simple_counter():
    print(f"收到數字: {number}")


- 實作：
    - 產線上一台機台一整天產出的 Text Log 有可能高達數 GB。如果用 open().readlines() 一口氣將幾百萬行讀入記憶體（RAM），伺服器會直接 OOM (Out of Memory) 倒地。必須改用「產出一行、處理一行」的 Lazy Evaluation（惰性求值）策略。
    - 腳本：


In [ ]:
import os
import sys

# 先動態建立一個模擬的巨大機台 Log 檔 (含 10 萬行資料)
mock_log_path = "mock_heavy_device.log"
with open(mock_log_path, "w", encoding="utf-8") as f:
    for i in range(100000):
        level = "INFO" if i % 100 != 0 else "ERROR"
        f.write(f"2026-08-04 08:00:{i % 60:02d} [{level}] Sensor_ID_04=0.23{i} \n")


# 實作生成器 (Yield)
def log_line_streamer(file_path):
    """
    使用 yield 的生成器：每次呼叫只從硬碟讀取並回傳一行文字，極大化節省 RAM 空間。
    """
    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            yield line.strip()


# 效能測試與記憶體驗證
log_gen = log_line_streamer(mock_log_path)

print(f"生成器物件本身的記憶體佔用: {sys.getsizeof(log_gen)} bytes (極小！)")

# 模擬產線分析：我們只撈取含 [ERROR] 的事件進行監控
error_counter = 0
for raw_line in log_gen:  # 迭代器運作：此時才會一行行跟硬碟要資料
    if "[ERROR]" in raw_line:
        error_counter += 1
        if error_counter <= 3:
            print(f"補獲即時異常日誌: {raw_line}")

print(f"統計完成：在此大數據日誌中共偵測到 {error_counter} 個 Error 事件。")

# 清理模擬檔案
if os.path.exists(mock_log_path):
    os.remove(mock_log_path)


## 4.**裝飾器 (Decorator)** ── **自動日誌記錄與計時重試機制**

- 語法：
    - 本質上是一個「外掛程式」。它可以讓你不用修改原本函式的程式碼，就能在原本函式執行前或執行後，自動加上額外的功能（例如自動計時、檢查權限）。
    - 語法上使用 @裝飾器名稱 放在目標函式的上方。
    - 裝飾器的結構像洋蔥，共有三層：外層傳參數、中層傳函式、內層傳引數。


In [ ]:
# 自訂一個外掛打招呼功能
def my_decorator(func):
    def wrapper():
        print("[外掛啟動] 執行函式前要做的事...")
        func()  # 真正執行原本的函式 def say_hello()
        print("[外掛結束] 執行函式後要做的事...")

    return wrapper  # 傳回原本的 say_hello


@my_decorator
def say_hello():
    print("Hello World! (這是原本的業務邏輯)")


# 呼叫原本的函式，會自動帶上外掛功能
say_hello()  # 此時不會去執行原本那個最陽春的 def say_hello():，而是會去執行被裝飾器替換掉的全新版本（也就是那個 wrapper 函數）


- 多重裝飾器
    - 當一個函式被同時疊上多個裝飾器時，它的執行順序是「由上往下（外到內）進入，再由內往外出」。


In [ ]:
def decorator_A(func):
    def wrapper():
        print("[A] 進入外層")
        func()
        print("[A] 離開外層")

    return wrapper


def decorator_B(func):
    def wrapper():
        print("[B] 進入內層")
        func()
        print("[B] 離開內層")

    return wrapper


# 同時疊加 A 與 B
@decorator_A
@decorator_B
def my_core_logic():
    print("執行業務邏輯")


my_core_logic()

""" 輸出
[A] 進入外層
  [B] 進入內層
     執行業務邏輯
  [B] 離開內層
[A] 離開外層 """

- 實作：柔性整合工程師寫的底層控制碼，必須要具備無痛「外掛」功能。例如每次撈取測試結果（Yield Data）時都要「自動紀錄執行時間」和「**失敗自動重試 3 次（Retry）**」。裝飾器能讓我們不用改動業務邏輯本體，直接優雅注入這些機制。
- 腳本：
    -


In [ ]:
import functools  # 保留被裝飾函式的原始屬性（如名稱、註解）。
import time


def retry_and_timer(max_attempts=3, delay=1):
    """定義帶參數的裝飾器：結合計時器與自動重試機制"""
    # 第一層：最大重試 3 次，每次失敗後隔 1 秒再試。

    def decorator(func):
        # 第二層：接收被裝飾的函式，並返回包裹後的函式
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # 第三層：包裹被裝飾函式的邏輯，實作重試與計時功能
            attempts = 0
            start_time = time.time()

            while attempts < max_attempts:
                try:
                    attempts += 1
                    result = func(
                        *args, **kwargs
                    )  # *args 和 **kwargs 代表可以接收目標函式的任何參數。
                    # 如果 func 成功執行（沒有拋出異常）
                    end_time = time.time()  # 記錄成功的時間點
                    print(
                        f"[{func.__name__}] 執行成功，耗時: {end_time - start_time:.4f} 秒"
                    )
                    return result
                # 如果 func 執行失敗（拋出異常）
                except Exception as e:
                    print(f"[{func.__name__}] 第 {attempts} 次失敗: {e}")
                    if attempts >= max_attempts:
                        print("已達最大重試次數，拋出異常。")
                        raise  # 這裡後面什麼都沒接，代表「把剛剛捕捉到的錯誤原封不動地再度拋出去」，讓更外層的程式去崩潰或處理。
                    time.sleep(delay)

        return wrapper

    return decorator


# 應用在硬體測試函式
@retry_and_timer(max_attempts=3, delay=0.5)
def read_wafer_id_sensor():
    """模擬讀取晶圓 ID 的感測器"""
    # 模擬現實世界不穩定的硬體環境。有 70% 的高機率會觸發 RuntimeError，30% 機率完美讀取。
    if random.random() > 0.3:  # 70% 機率失敗
        raise RuntimeError("感測器光學鏡頭被遮擋，讀取失敗")
    return "WAFER_LOT1234_DIE01"


# 測試執行
try:
    wafer_id = read_wafer_id_sensor()
    print(f"最終獲取數據: {wafer_id}")
# 連錯 3 次
except Exception:
    print("自動化 Pipeline 流程中斷，觸發警報。")

## 6.**上下文管理器 (Context Managers)**

- 語法：
    - 最常見的 with 用法是讀寫檔案（例如 with open(...)）。
      只要離開 with 的縮排區塊，Python 就會自動幫你關閉資源，就算中間程式當機也一樣。
    - 要自訂這個行為，類別（Class）裡面必須內建兩個特殊方法：
    - **enter**：進入 with 區塊時，程式「自動」要做的事情（例如：開啟連線）。
    - **exit**：離開 with 區塊時，程式「絕對」會做的事情（例如：關閉連線、釋放記憶體）。


In [ ]:
# 自訂一個開關
class SimpleSwitch:
    def __enter__(self):
        print(" [自動執行] 開關開啟了！")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        print(" [自動執行] 離開區塊了，開關安全關閉！")


# 測試使用
with SimpleSwitch():
    print(" 正在執行區塊內部的程式碼...")


- 實作：
    - 除了基本的 with open() 外，自動化測試整合最常遇到「資料庫連線管理」與「測試儀器連線釋放」。我們用 enter 與 exit 來自訂安全的設備連線管理器。
    - 透過類別實作 **enter** 與 **exit**，確保測試儀器不論成功或失敗，都能自動斷開連線、釋放資源，防止通訊埠（Socket）被鎖死。
    - 腳本：
        - exc_type：異常的類型（Exception Type），例如 ValueError 或 ConnectionError。
        - exc_val：異常的值（Exception Value），也就是錯誤訊息本身。
        - exc_tb：追蹤物件（Traceback），記錄發生錯誤時的程式碼行號與呼叫堆疊。
        - raise RuntimeError()：拋出錯誤。


In [ ]:
class ProbeCardConnectionManager:
    """自訂半導體探針台連線上下文管理器"""

    def __init__(self, ip_address: str):
        self.ip_address = ip_address
        self.socket_port = None

    def __enter__(self):
        # 進入 with 區塊時自動執行：初始化通訊埠並建立連線
        print(f"[連線初始化] 正在開啟 Socket 連線至探針台 ({self.ip_address})...")
        self.socket_port = 5025  # 模擬分配的硬體 Port
        return self  # 回傳物件本身給 as 後面的變數

    def send_scpi_command(self, command: str):
        """模擬發送 SCPI 指令控制硬體"""
        if "ERROR" in command:
            raise RuntimeError("儀器內部電壓超載，拒絕執行指令。")
        print(f"[指令發送] 成功寫入指令: {command}")
        return "SUCCESS"

    def __exit__(self, exc_type, exc_val, exc_tb):
        # 不管 with 區塊內是否發生錯誤，離開時「一定會執行」的清理通道
        print(
            f"[安全關閉] 正在關閉 Socket Port: {self.socket_port}，釋放硬體控制權。"
        )
        self.socket_port = None

        if exc_type:
            print(f"[異常捕獲] 偵測到內部執行錯誤: {exc_val}。異常已安全隔離。")
            return True  # 回傳 True 代表在這一層化解錯誤，不讓程式崩潰中斷


# 實戰測試隨插即用
with ProbeCardConnectionManager("192.168.1.50") as probe:
    probe.send_scpi_command("*IDN?")
    # 模擬觸發異常，測試 __exit__ 是否仍能安全釋放硬體
    probe.send_scpi_command("MEASure:VOLTage:DC? TRIGGER_ERROR")

print("自動化 Pipeline 繼續平穩運行。")